# 00 - Data Preprocessing

Resize the Anime Face Dataset to **256x256** and group into a single processed folder. Output: `{ROOT}/data/processed/anime_256` - the canonical image set the rest of the pipeline reads.

**Chain:** raw `data/anime_images/*` -> `data/processed/anime_256/*`.

In [ ]:

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass

Mounted at /content/drive


In [ ]:
import os, glob, json, math, textwrap, random, re
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE=='cuda' else torch.float32
ROOT = next((r for r in ['/content/drive/MyDrive/Text2ImageNarration',
                         '/content/drive/MyDrive/AnimeStoryGen','.'] if os.path.isdir(r)), '.')
DATA = f'{ROOT}/data'; PROC = f'{ROOT}/data/processed/anime_256'
ART  = f'{ROOT}/artifacts'; CKPT = f'{ROOT}/checkpoints'
GEN  = f'{ROOT}/outputs/generated'; BASE = f'{ROOT}/baselines/xlxmert'; EVALD = f'{ROOT}/outputs/eval'
for d in (DATA, ART, CKPT, GEN, EVALD): os.makedirs(d, exist_ok=True)
SEED = 42; N_TEST = 200
def img_dir():
    return PROC if (os.path.isdir(PROC) and glob.glob(f'{PROC}/*')) else f'{DATA}/anime_images'
from PIL import Image, ImageDraw, ImageFont
print('device', DEVICE, '| ROOT', ROOT, '| images', img_dir())

device cuda | ROOT /content/drive/MyDrive/Text2ImageNarration | images /content/drive/MyDrive/Text2ImageNarration/data/anime_images


In [ ]:
# Resize + group (resumable: skips files already processed)
SRC = f'{DATA}/anime_images'
os.makedirs(PROC, exist_ok=True)
files = sorted(glob.glob(f'{SRC}/*'))
print(len(files), 'source images at', SRC)
done = skip = 0
for f in files:
    out = f'{PROC}/{os.path.basename(f)}'
    if os.path.exists(out): skip += 1; continue
    try:
        Image.open(f).convert('RGB').resize((256,256), Image.LANCZOS).save(out); done += 1
    except Exception as e: print('skip', f, e)
print(f'resized {done} | already {skip} | total {len(glob.glob(PROC+chr(47)+chr(42)))} -> {PROC}')

55 source images at /content/drive/MyDrive/Text2ImageNarration/data/anime_images
resized 55 | already 0 | total 55 -> /content/drive/MyDrive/Text2ImageNarration/data/processed/anime_256


**Next:** `01_ViT_Captioning_and_Split`.